In [ ]:
import pickle as pkl

import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
import numpy as np
norm_dict = {"max":max,"min":min,"None":lambda x:1}



### Open the saved distances

In [ ]:
with open("file/to/path.pkl", "rb") as f:
    file = pkl.load(f)

### Add the parameters

In [ ]:
time = 0
normalization= "max"

In [ ]:
def plot_clustermap(file,time,normalization):
    comparisons = file["comparisons"]
    names = file["names"]
    norms = file["norms"]
    manager = file["ltm"]

    len_all_trees = len(names[time].keys())
    hierarchy = np.zeros((len_all_trees, len_all_trees))
    labels_lT = [names[time][n][0] for n in names[time]]
    labels_root = [names[time][n][2] for n in names[time]]
    labels_node = [names[time][n][1] for n in names[time]]
    labels = [
       manager.lineagetrees[names[time][n][0]].labels[
             manager.lineagetrees[
                names[time][n][0]
            ].get_labelled_ancestor(names[time][n][1])
        ]
        for n in names[time]
    ]
    labels_of_clustermap = [
        names[time][n][0] + "_" + str(labels[n])
        for n in names[time]
    ]
    for keys, values in comparisons[time]:
        hierarchy[keys, values] = comparisons[time][
            keys, values
        ] / norm_dict[normalization](
            norms[time][keys, values]
        )
        hierarchy[values, keys] = hierarchy[keys, values]

    condensed_dist_matrix = squareform(hierarchy)

    linkage_data = linkage(condensed_dist_matrix, method="ward")
    order = dendrogram(linkage_data, no_plot=True)["leaves"]
    labels_of_clustermap = [
        labels_of_clustermap[i] for i in order
    ]
    labels_lT = [labels_lT[i] for i in order]
    labels_node = [labels_node[i] for i in order]
    labels_root = [labels_root[i] for i in order]
    fig,ax = plt.subplots()
    plot = hierarchy[np.ix_(order, order)]

    plot = hierarchy[np.ix_(order, order)]
    plot = ax.imshow(
        plot, cmap="viridis"
    )
    fig.colorbar(plot, ax=ax)
    ax.set_xticks(
        np.arange(len(labels_of_clustermap)),
        labels=labels_of_clustermap,
    )
    ax.set_yticks(
        np.arange(len(labels_of_clustermap)),
        labels=labels_of_clustermap,
    )
    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
    )

### The results

In [ ]:
plot_clustermap(file,time,normalization)